# Limpieza de datos: jugadores de fútbol (ValorGol)

**Curso:** Algoritmos y Programación, UdeA 2026-2

## 1. Contexto
- **Fuente original:** *Transfermarkt datasets*, publicado por David Cariboo en Kaggle (`davidcariboo/player-scores`, https://www.kaggle.com/datasets/davidcariboo/player-scores) y en GitHub (`dcaribou/transfermarkt-datasets`). Descargado en noviembre de 2026. Los datos originales provienen del sitio Transfermarkt
- **Licencia:** CC0 1.0 (dominio público)
- **Temática:** jugadores de fútbol profesional y su valor de mercado.
- **Archivo original:** `data/raw/players.csv`
- **Objetivo:** dejar un CSV limpio (`data/datos.csv`) que alimente la app: filtros por posición, país, edad y valor, y el juego «¿Cuánto vale?».

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RUTA_RAW = Path("raw") / "players.csv"     # el cuaderno se ejecuta desde la carpeta data/
RUTA_SALIDA = Path("datos.csv")
ANIO_ACTUAL = 2026

## 2. Carga y revisión inicial

In [2]:
df_raw = pd.read_csv(RUTA_RAW)
print("Filas y columnas:", df_raw.shape)
df_raw.head()

Filas y columnas: (50149, 26)


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus FC,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,Bayern Munich,100000.0,3000000.0


In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 50149 entries, 0 to 50148
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   player_id                             50149 non-null  int64  
 1   first_name                            46985 non-null  str    
 2   last_name                             50149 non-null  str    
 3   name                                  50149 non-null  str    
 4   last_season                           50149 non-null  int64  
 5   current_club_id                       50149 non-null  int64  
 6   player_code                           50149 non-null  str    
 7   country_of_birth                      44264 non-null  str    
 8   city_of_birth                         44541 non-null  str    
 9   country_of_citizenship                49880 non-null  str    
 10  date_of_birth                         50100 non-null  str    
 11  sub_position              

In [4]:
# Porcentaje de valores faltantes por columna
(df_raw.isna().mean() * 100).round(1).sort_values(ascending=False)

current_national_team_id                93.5
international_caps                      61.4
international_goals                     61.4
agent_name                              46.5
contract_expiration_date                37.0
highest_market_value_in_eur             17.2
market_value_in_eur                     17.2
foot                                    11.8
country_of_birth                        11.7
city_of_birth                           11.2
height_in_cm                             8.7
first_name                               6.3
current_club_domestic_competition_id     6.0
current_club_name                        6.0
sub_position                             1.2
country_of_citizenship                   0.5
date_of_birth                            0.1
player_id                                0.0
player_code                              0.0
name                                     0.0
last_season                              0.0
current_club_id                          0.0
last_name 

## 3. Decisiones de limpieza
| # | Decisión | Justificación |
|---|---|---|
| 1 | Eliminar duplicados por `player_id` | Un jugador no debe aparecer dos veces. |
| 2 | Conservar solo jugadores activos (`last_season` dentro de las dos últimas temporadas del archivo) | Los retirados distorsionan el valor y el juego. |
| 3 | Eliminar jugadores sin valor de mercado o con valor 0 | El valor es la variable central de la app. |
| 4 | Calcular `edad` desde `date_of_birth` | La app filtra y grafica por edad. |
| 5 | Limpiar espacios en textos | Evita categorías repetidas por errores de escritura. |
| 6 | Alturas fuera de 150–215 cm pasan a nulo | Son errores de captura. |
| 7 | Rellenar `pie` y `país` faltantes con «Desconocido» | La app no debe fallar con vacíos. |
| 8 | Descartar posiciones sin clasificar y traducir posición y pie al español | Evita categorías vacías y mejora la lectura para el usuario. |
| 9 | Seleccionar solo las columnas necesarias | Reduce el tamaño del archivo. |

In [5]:
df = df_raw.copy()

# 1. Duplicados
antes = len(df)
df = df.drop_duplicates(subset="player_id")
print(f"Duplicados eliminados: {antes - len(df)}")

# 2. Jugadores activos
antes = len(df)
ultima_temporada = df["last_season"].max()
df = df[df["last_season"] >= ultima_temporada - 1]
print(f"Última temporada en el archivo: {ultima_temporada}")
print(f"Jugadores no activos eliminados: {antes - len(df)}")

# 3. Valor de mercado válido
antes = len(df)
df = df.dropna(subset=["market_value_in_eur"])
df = df[df["market_value_in_eur"] > 0]
print(f"Sin valor de mercado eliminados: {antes - len(df)}")

# 4. Fecha de nacimiento y edad
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], errors="coerce")
df = df.dropna(subset=["date_of_birth"])
df["edad"] = ((pd.Timestamp.today() - df["date_of_birth"]).dt.days // 365.25).astype(int)

# 5. Textos sin espacios sobrantes
for col in ["name", "country_of_citizenship", "current_club_name"]:
    df[col] = df[col].astype("string").str.strip()

# 6. Alturas imposibles
df.loc[(df["height_in_cm"] < 150) | (df["height_in_cm"] > 215), "height_in_cm"] = np.nan

# 7. Faltantes categóricos
df["foot"] = df["foot"].fillna("Desconocido")
df["country_of_citizenship"] = df["country_of_citizenship"].fillna("Desconocido")

# 8. Solo posiciones válidas y traducciones
df = df[df["position"].isin(["Attack", "Midfield", "Defender", "Goalkeeper"])]
df["position"] = df["position"].replace({
    "Attack": "Delantero", "Midfield": "Mediocampista",
    "Defender": "Defensa", "Goalkeeper": "Portero",
})
df["foot"] = df["foot"].replace({
    "right": "Derecho", "left": "Izquierdo", "both": "Ambidiestro",
})

print("Filas tras la limpieza:", len(df))

Duplicados eliminados: 0
Última temporada en el archivo: 2025
Jugadores no activos eliminados: 21782
Sin valor de mercado eliminados: 7460
Filas tras la limpieza: 20876


## 4. Selección y renombrado de columnas

In [6]:
columnas = {
    "player_id": "player_id",
    "name": "nombre",
    "position": "posicion",
    "sub_position": "subposicion",
    "country_of_citizenship": "pais",
    "current_club_name": "club",
    "edad": "edad",
    "height_in_cm": "altura_cm",
    "foot": "pie",
    "market_value_in_eur": "valor_eur",
    "highest_market_value_in_eur": "valor_max_eur",
    "image_url": "imagen_url",
}
df_final = df[list(columnas)].rename(columns=columnas).reset_index(drop=True)
df_final.head()

,player_id,nombre,posicion,subposicion,pais,club,edad,altura_cm,pie,valor_eur,valor_max_eur,imagen_url
0,2989,Paolo Guerrero,Delantero,Centre-Forward,Peru,<NA>,42,185.0,Derecho,50000.0,8000000.0,https://img.a.transfermarkt.technology/portrai...
1,3333,James Milner,Mediocampista,Central Midfield,England,Brighton & Hove Albion,40,175.0,Derecho,500000.0,21000000.0,https://img.a.transfermarkt.technology/portrai...
2,3755,Óscar Ustari,Portero,Goalkeeper,Argentina,Club Internacional de Fútbol Miami,40,183.0,Derecho,600000.0,7000000.0,https://img.a.transfermarkt.technology/portrai...
3,5336,Anastasios Tsokanis,Mediocampista,Defensive Midfield,Greece,Volou Neos Podosferikos Syllogos,35,176.0,Izquierdo,150000.0,450000.0,https://img.a.transfermarkt.technology/portrai...
4,7161,Jonas Hofmann,Mediocampista,Attacking Midfield,Germany,Bayer 04 Leverkusen,34,176.0,Derecho,1500000.0,16000000.0,https://img.a.transfermarkt.technology/portrai...


## 5. Exploración rápida del resultado

In [7]:
print(df_final.describe(include="all").T[["count", "unique", "mean", "min", "max"]])
df_final["posicion"].value_counts()

                 count unique            mean      min          max
player_id      20876.0    NaN   603608.999281   2989.0    1523289.0
nombre           20876  20568             NaN      NaN          NaN
posicion         20876      4             NaN      NaN          NaN
subposicion      20876     13             NaN      NaN          NaN
pais             20876    176             NaN      NaN          NaN
club             18684    667             NaN      NaN          NaN
edad           20876.0    NaN        26.61027     16.0         46.0
altura_cm      20095.0    NaN       182.13063    155.0        210.0
pie              20876      4             NaN      NaN          NaN
valor_eur      20876.0    NaN  2712613.048477  10000.0  200000000.0
valor_max_eur  20876.0    NaN  4976857.875072  10000.0  200000000.0
imagen_url       20876  19477             NaN      NaN          NaN


posicion
Defensa          6827
Mediocampista    5953
Delantero        5798
Portero          2298
Name: count, dtype: int64

## 6. Validaciones y exportación

In [8]:
assert len(df_final) > 1000, "El dataset debe tener más de 1.000 registros"
assert df_final["valor_eur"].min() > 0
assert df_final["player_id"].is_unique

df_final.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"Guardado {RUTA_SALIDA} con {len(df_final):,} filas")
print(f"Tamaño: {RUTA_SALIDA.stat().st_size / 1e6:.1f} MB (meta: menos de 50 MB)")

Guardado datos.csv con 20,876 filas
Tamaño: 4.0 MB (meta: menos de 50 MB)


## 7. Diccionario de variables
| Variable | Descripción |
|---|---|
| `player_id` | Identificador único del jugador |
| `nombre` | Nombre completo |
| `posicion` | Portero, Defensa, Mediocampista o Delantero |
| `subposicion` | Posición específica (extremo, central, etc.) |
| `pais` | Nacionalidad |
| `club` | Club actual |
| `edad` | Edad en años |
| `altura_cm` | Altura en centímetros |
| `pie` | Pie dominante |
| `valor_eur` | Valor de mercado actual en euros |
| `valor_max_eur` | Valor de mercado máximo histórico |
| `imagen_url` | Enlace a la foto del jugador |

> Si los nombres de columnas del archivo original son distintos, ajusta el diccionario `columnas` y las celdas de limpieza.